# PowerGrid Utility Intelligence
## Predictive Grid Failure Analytics — Moving from Reactive to Proactive Operations

**Capstone Project | IITM–JARO Batch 02**  
**Submitted to:** Prof. Babji Srinivasan  
**Dataset:** 50,500 records | 32 features

---

## Objective
Develop a binary classification model to predict grid failures and generate asset-level risk-tier scores to transition PowerGrid from **reactive** to **proactive** maintenance strategy.

**Key Deliverables:**
1. High-recall binary classification model
2. SHAP explainability analysis
3. Asset risk-tier scoring (High/Medium/Low)
4. Business recommendations & ROI analysis

---
## Phase 1: Environment Setup & Data Loading
**Week 1 - Data Understanding & Profiling**

In [ ]:
# Import core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ML & preprocessing
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.imbalance import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, roc_curve,
    precision_recall_curve, f1_score, recall_score, precision_score
)

# Advanced models
try:
    import xgboost as xgb
except ImportError:
    print("XGBoost not installed. Install with: pip install xgboost")

try:
    import lightgbm as lgb
except ImportError:
    print("LightGBM not installed. Install with: pip install lightgbm")

# Explainability
try:
    import shap
except ImportError:
    print("SHAP not installed. Install with: pip install shap")

# Plotting preferences
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)

print("✓ All libraries imported successfully!")

### Load Dataset
Upload your CSV file (50,500 records × 32 features) or load from your local path.

In [ ]:
# OPTION 1: Load from local file path
# df = pd.read_csv('path/to/your/powergrid_data.csv')

# OPTION 2: Load from URL (if available)
# df = pd.read_csv('https://your-url/powergrid_data.csv')

# For now, create a sample dataset structure to demonstrate the pipeline
# Replace this with your actual data load once file is available
print("⚠️  PLACEHOLDER: Replace with your actual data")
print("\nExpected dataset structure:")
print("Shape: (50500, 32)")
print("\nDataset domains:")
print("  • Equipment health: health_score, transformer_temp, oil_quality, vibration")
print("  • Operational telemetry: load_mw, utilisation_pct, voltage_fluctuation, frequency_deviation")
print("  • Maintenance history: overdue_days, maintenance_events_12m, inspection_score, prior_outages")
print("  • Weather & environment: temperature, humidity, wind_speed, rainfall, storm_risk_index")
print("  • Business impact: customers_served, revenue_loss_estimate, regulatory_penalty")
print("  • Legacy/admin: legacy_asset_code, monitoring_batch_id, administrative_reference (TO DROP)")
print("\nTarget variable: grid_failure_flag (0/1 - 57% no-failure, 43% failure)")

In [ ]:
# ===== LOAD YOUR DATA HERE =====
# Uncomment and modify the line below to load your actual dataset:
# df = pd.read_csv('your_powergrid_dataset.csv')

# For demonstration, we'll use sample code structure:
# After loading, run the next cell to proceed

# df.head()
# df.shape
# df.info()

### Data Profiling Report

In [ ]:
def data_profiling_report(df):
    """
    Generate comprehensive data profiling report
    """
    print("="*80)
    print("DATA PROFILING REPORT")
    print("="*80)
    
    print(f"\n📊 DATASET SHAPE: {df.shape}")
    print(f"   Rows: {df.shape[0]:,}")
    print(f"   Columns: {df.shape[1]}")
    
    print(f"\n📋 COLUMN TYPES:")
    print(df.dtypes.value_counts())
    
    print(f"\n❌ MISSING VALUES:")
    missing = df.isnull().sum()
    if missing.sum() == 0:
        print("   ✓ No missing values detected!")
    else:
        print(missing[missing > 0])
    
    print(f"\n⚖️  CLASS BALANCE (target: grid_failure_flag):")
    if 'grid_failure_flag' in df.columns:
        class_dist = df['grid_failure_flag'].value_counts(normalize=True) * 100
        print(class_dist)
        print(f"\n   ⚠️  Class Imbalance Ratio: {class_dist[0]:.1f}% vs {class_dist[1]:.1f}%")
        print("   → Will apply SMOTE oversampling in preprocessing phase")
    
    print(f"\n📈 NUMERICAL COLUMNS SUMMARY:")
    print(df.describe())
    
    print(f"\n🔤 CATEGORICAL COLUMNS:")
    cat_cols = df.select_dtypes(include='object').columns
    for col in cat_cols:
        print(f"\n   {col}: {df[col].nunique()} unique values")
        print(f"   {df[col].value_counts().to_dict()}")
    
    print(f"\n⚙️  DATA QUALITY ISSUES DETECTED:")
    issues = []
    
    # Check for inconsistent casing in categorical columns
    for col in cat_cols:
        if df[col].str.lower().nunique() < df[col].nunique():
            issues.append(f"   ⚠️  '{col}' has inconsistent casing")
    
    # Check for legacy columns to drop
    legacy_cols = ['legacy_asset_code', 'monitoring_batch_id', 'administrative_reference']
    for col in legacy_cols:
        if col in df.columns:
            issues.append(f"   ⚠️  '{col}' flagged for exclusion (no predictive value)")
    
    if issues:
        for issue in issues:
            print(issue)
    else:
        print("   ✓ No data quality issues detected")
    
    print("\n" + "="*80 + "\n")

# Call the profiling function when data is loaded
# data_profiling_report(df)

---
## Phase 2: Exploratory Data Analysis (EDA)
**Week 2 - Understanding Patterns & Relationships**

In [ ]:
def exploratory_data_analysis(df):
    """
    Comprehensive EDA with visualizations
    """
    
    # 1. Target variable distribution
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    df['grid_failure_flag'].value_counts().plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c'])
    axes[0].set_title('Grid Failure Flag Distribution (Count)', fontsize=12, fontweight='bold')
    axes[0].set_ylabel('Number of Assets')
    axes[0].set_xticklabels(['No Failure (0)', 'Failure (1)'], rotation=0)
    
    df['grid_failure_flag'].value_counts(normalize=True).plot(kind='pie', ax=axes[1], 
                                                                autopct='%1.1f%%',
                                                                colors=['#2ecc71', '#e74c3c'])
    axes[1].set_title('Class Distribution (%)', fontsize=12, fontweight='bold')
    axes[1].set_ylabel('')
    plt.tight_layout()
    plt.show()
    
    # 2. Numerical features distribution
    numerical_cols = df.select_dtypes(include=[np.number]).columns
    numerical_cols = [col for col in numerical_cols if col != 'grid_failure_flag']
    
    # Plot distributions for first 12 numerical features
    n_features = min(12, len(numerical_cols))
    fig, axes = plt.subplots(4, 3, figsize=(16, 12))
    axes = axes.flatten()
    
    for idx, col in enumerate(numerical_cols[:n_features]):
        axes[idx].hist(df[col], bins=50, color='skyblue', edgecolor='black', alpha=0.7)
        axes[idx].set_title(f'{col}', fontweight='bold', fontsize=10)
        axes[idx].set_ylabel('Frequency')
    
    plt.tight_layout()
    plt.show()
    
    # 3. Correlation heatmap (top features)
    plt.figure(figsize=(14, 10))
    corr_matrix = df[numerical_cols[:15]].corr()
    sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, 
                square=True, cbar_kws={"shrink": 0.8})
    plt.title('Correlation Matrix - Top 15 Numerical Features', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # 4. Failure rate by key categorical feature (if asset_type exists)
    if 'asset_type' in df.columns:
        failure_by_type = df.groupby('asset_type')['grid_failure_flag'].agg(['sum', 'count', 'mean'])
        failure_by_type.columns = ['Failures', 'Total', 'Failure_Rate']
        failure_by_type['Failure_Rate'] = failure_by_type['Failure_Rate'] * 100
        
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        failure_by_type['Total'].plot(kind='bar', ax=axes[0], color='skyblue')
        axes[0].set_title('Asset Distribution by Type', fontsize=12, fontweight='bold')
        axes[0].set_ylabel('Count')
        
        failure_by_type['Failure_Rate'].plot(kind='bar', ax=axes[1], color='coral')
        axes[1].set_title('Failure Rate by Asset Type', fontsize=12, fontweight='bold')
        axes[1].set_ylabel('Failure Rate (%)')
        plt.tight_layout()
        plt.show()
        
        print("\n📊 Failure Rate by Asset Type:")
        print(failure_by_type)
    
    # 5. Top features correlated with failures
    print("\n🔍 Top 10 Features Correlated with Grid Failure:")
    failure_corr = df[numerical_cols + ['grid_failure_flag']].corr()['grid_failure_flag'].drop('grid_failure_flag')
    top_features = failure_corr.abs().nlargest(10)
    print(failure_corr[top_features.index])
    
    # 6. Box plots for top failure predictors
    top_pred_cols = failure_corr.abs().nlargest(4).index
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    axes = axes.flatten()
    
    for idx, col in enumerate(top_pred_cols):
        df.boxplot(column=col, by='grid_failure_flag', ax=axes[idx])
        axes[idx].set_title(f'{col} by Grid Failure Status', fontsize=10, fontweight='bold')
        axes[idx].set_xlabel('Grid Failure (0=No, 1=Yes)')
    
    plt.suptitle('Top Failure Predictors - Distribution Analysis', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

# Call EDA when data is loaded
# exploratory_data_analysis(df)

---
## Phase 3: Data Preprocessing & Feature Engineering
**Week 3 - Data Cleaning, Encoding, Balancing**

In [ ]:
def preprocess_and_engineer_features(df):
    """
    Complete preprocessing pipeline:
    1. Drop legacy columns
    2. Normalize categorical data
    3. Handle missing values
    4. Create interaction features
    5. Encode categorical variables
    6. Scale numerical features
    7. Apply SMOTE for class imbalance
    """
    
    print("\n" + "="*80)
    print("PREPROCESSING & FEATURE ENGINEERING PIPELINE")
    print("="*80)
    
    df = df.copy()
    
    # ========== STEP 1: DROP LEGACY COLUMNS ==========
    print("\n[STEP 1] Dropping legacy columns...")
    legacy_cols = ['legacy_asset_code', 'monitoring_batch_id', 'administrative_reference']
    cols_to_drop = [col for col in legacy_cols if col in df.columns]
    if cols_to_drop:
        df = df.drop(columns=cols_to_drop)
        print(f"   ✓ Dropped: {cols_to_drop}")
    else:
        print("   ✓ No legacy columns found")
    
    # ========== STEP 2: NORMALIZE CATEGORICAL DATA ==========
    print("\n[STEP 2] Normalizing categorical data...")
    for col in df.select_dtypes(include='object').columns:
        if col != 'grid_failure_flag':  # Don't normalize target
            before = df[col].nunique()
            df[col] = df[col].str.strip().str.lower()
            after = df[col].nunique()
            if before != after:
                print(f"   ✓ '{col}': {before} → {after} unique values")
    
    # ========== STEP 3: HANDLE MISSING VALUES ==========
    print("\n[STEP 3] Handling missing values...")
    missing = df.isnull().sum()
    if missing.sum() > 0:
        numerical_cols = df.select_dtypes(include=[np.number]).columns
        for col in numerical_cols:
            if df[col].isnull().sum() > 0:
                df[col].fillna(df[col].median(), inplace=True)
        
        categorical_cols = df.select_dtypes(include='object').columns
        for col in categorical_cols:
            if df[col].isnull().sum() > 0:
                df[col].fillna(df[col].mode()[0], inplace=True)
        print(f"   ✓ Missing values imputed")
    else:
        print("   ✓ No missing values detected")
    
    # ========== STEP 4: CREATE INTERACTION FEATURES ==========
    print("\n[STEP 4] Engineering interaction features...")
    numerical_cols = df.select_dtypes(include=[np.number]).columns
    
    # Example interactions (modify based on your domain understanding)
    interaction_features = {}
    
    # Common interactions for power grid
    if 'load_mw' in df.columns and 'utilisation_pct' in df.columns:
        df['load_x_utilisation'] = df['load_mw'] * df['utilisation_pct']
        print("   ✓ Created: load_x_utilisation")
    
    if 'transformer_temp' in df.columns and 'oil_quality' in df.columns:
        df['temp_x_oil_quality'] = df['transformer_temp'] * df['oil_quality']
        print("   ✓ Created: temp_x_oil_quality")
    
    # Prepare features and target
    target = df['grid_failure_flag']
    X = df.drop(columns=['grid_failure_flag'])
    
    # ========== STEP 5: ENCODE CATEGORICAL VARIABLES ==========
    print("\n[STEP 5] Encoding categorical variables...")
    categorical_cols = X.select_dtypes(include='object').columns
    
    for col in categorical_cols:
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col])
    print(f"   ✓ Encoded {len(categorical_cols)} categorical features")
    
    # ========== STEP 6: SCALE NUMERICAL FEATURES ==========
    print("\n[STEP 6] Scaling numerical features...")
    scaler = StandardScaler()
    numerical_features = X.select_dtypes(include=[np.number]).columns
    X[numerical_features] = scaler.fit_transform(X[numerical_features])
    print(f"   ✓ Scaled {len(numerical_features)} numerical features")
    
    # ========== STEP 7: HANDLE CLASS IMBALANCE WITH SMOTE ==========
    print("\n[STEP 7] Applying SMOTE for class balancing...")
    print(f"   Before SMOTE: {target.value_counts().to_dict()}")
    
    smote = SMOTE(random_state=42, k_neighbors=5)
    X_balanced, y_balanced = smote.fit_resample(X, target)
    
    print(f"   After SMOTE: {pd.Series(y_balanced).value_counts().to_dict()}")
    print(f"   ✓ Dataset balanced: {len(X_balanced)} samples")
    
    print("\n" + "="*80)
    print(f"\n✅ Preprocessing complete!")
    print(f"   Final shape: {X_balanced.shape}")
    print(f"   Target distribution: {pd.Series(y_balanced).value_counts(normalize=True).to_dict()}")
    
    return X_balanced, y_balanced, scaler

# Usage:
# X_balanced, y_balanced, scaler = preprocess_and_engineer_features(df)

---
## Phase 4: Model Development
**Week 4 - Training Multiple Classifiers**

In [ ]:
def train_models(X_balanced, y_balanced):
    """
    Train and compare multiple classifiers:
    - Logistic Regression (baseline)
    - Random Forest
    - XGBoost
    - LightGBM
    """
    
    print("\n" + "="*80)
    print("MODEL DEVELOPMENT & TRAINING")
    print("="*80)
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X_balanced, y_balanced, test_size=0.2, random_state=42, stratify=y_balanced
    )
    
    print(f"\n📊 Train-Test Split:")
    print(f"   Training set: {X_train.shape}")
    print(f"   Test set: {X_test.shape}")
    
    models = {}
    
    # ========== MODEL 1: LOGISTIC REGRESSION (BASELINE) ==========
    print("\n[MODEL 1] Training Logistic Regression (Baseline)...")
    lr = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1)
    lr.fit(X_train, y_train)
    models['Logistic Regression'] = lr
    print("   ✓ Training complete")
    
    # ========== MODEL 2: RANDOM FOREST ==========
    print("\n[MODEL 2] Training Random Forest...")
    rf = RandomForestClassifier(n_estimators=100, max_depth=15, min_samples_split=5,
                                 random_state=42, n_jobs=-1, class_weight='balanced')
    rf.fit(X_train, y_train)
    models['Random Forest'] = rf
    print("   ✓ Training complete")
    
    # ========== MODEL 3: XGBOOST ==========
    print("\n[MODEL 3] Training XGBoost...")
    try:
        xgb_model = xgb.XGBClassifier(
            n_estimators=100, max_depth=6, learning_rate=0.1,
            random_state=42, n_jobs=-1, scale_pos_weight=len(y_train[y_train==0])/len(y_train[y_train==1])
        )
        xgb_model.fit(X_train, y_train, eval_metric='logloss')
        models['XGBoost'] = xgb_model
        print("   ✓ Training complete")
    except Exception as e:
        print(f"   ⚠️  XGBoost training failed: {str(e)}")
    
    # ========== MODEL 4: LIGHTGBM ==========
    print("\n[MODEL 4] Training LightGBM...")
    try:
        lgb_model = lgb.LGBMClassifier(
            n_estimators=100, max_depth=7, learning_rate=0.1,
            random_state=42, n_jobs=-1, is_unbalance=True
        )
        lgb_model.fit(X_train, y_train)
        models['LightGBM'] = lgb_model
        print("   ✓ Training complete")
    except Exception as e:
        print(f"   ⚠️  LightGBM training failed: {str(e)}")
    
    print("\n" + "="*80)
    
    return models, X_train, X_test, y_train, y_test

# Usage:
# models, X_train, X_test, y_train, y_test = train_models(X_balanced, y_balanced)

---
## Phase 5: Model Evaluation & Explainability
**Week 5 - Performance Analysis & SHAP Explanations**

In [ ]:
def evaluate_models(models, X_test, y_test):
    """
    Comprehensive model evaluation:
    - Classification metrics (Precision, Recall, F1, AUC-ROC)
    - Confusion matrices
    - ROC-AUC curves
    """
    
    print("\n" + "="*80)
    print("MODEL EVALUATION REPORT")
    print("="*80)
    
    results = {}
    
    for model_name, model in models.items():
        print(f"\n🔍 {model_name}")
        print("-" * 60)
        
        # Predictions
        y_pred = model.predict(X_test)
        y_pred_proba = model.predict_proba(X_test)[:, 1]
        
        # Calculate metrics
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        auc_roc = roc_auc_score(y_test, y_pred_proba)
        
        results[model_name] = {
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'auc_roc': auc_roc,
            'y_pred': y_pred,
            'y_pred_proba': y_pred_proba
        }
        
        # Print metrics
        print(f"   Precision: {precision:.4f}")
        print(f"   Recall: {recall:.4f}")
        print(f"   F1-Score: {f1:.4f}")
        print(f"   AUC-ROC: {auc_roc:.4f}")
        
        # Confusion matrix
        cm = confusion_matrix(y_test, y_pred)
        print(f"\n   Confusion Matrix:")
        print(f"   True Negatives: {cm[0,0]}  |  False Positives: {cm[0,1]}")
        print(f"   False Negatives: {cm[1,0]}  |  True Positives: {cm[1,1]}")
    
    # ========== COMPARISON TABLE ==========
    print("\n" + "="*80)
    print("\n📊 MODEL COMPARISON")
    print("-" * 60)
    
    comparison_df = pd.DataFrame(results).T[['precision', 'recall', 'f1', 'auc_roc']]
    print(comparison_df.round(4))
    
    best_model_name = comparison_df['f1'].idxmax()
    print(f"\n🏆 Best Model (F1-Score): {best_model_name}")
    
    # ========== ROC-AUC CURVES ==========
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()
    
    for idx, (model_name, metrics) in enumerate(results.items()):
        if idx < 4:
            fpr, tpr, _ = roc_curve(y_test, metrics['y_pred_proba'])
            axes[idx].plot(fpr, tpr, color='darkorange', lw=2,
                           label=f"AUC = {metrics['auc_roc']:.4f}")
            axes[idx].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
            axes[idx].set_xlabel('False Positive Rate')
            axes[idx].set_ylabel('True Positive Rate')
            axes[idx].set_title(f'{model_name} - ROC-AUC Curve', fontweight='bold')
            axes[idx].legend(loc="lower right")
            axes[idx].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # ========== CONFUSION MATRICES ==========
    fig, axes = plt.subplots(1, len(results), figsize=(5*len(results), 4))
    if len(results) == 1:
        axes = [axes]
    
    for idx, (model_name, metrics) in enumerate(results.items()):
        cm = confusion_matrix(y_test, metrics['y_pred'])
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx], cbar=False)
        axes[idx].set_title(f'{model_name}', fontweight='bold')
        axes[idx].set_ylabel('True Label')
        axes[idx].set_xlabel('Predicted Label')
    
    plt.suptitle('Confusion Matrices - All Models', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()
    
    print("\n" + "="*80 + "\n")
    
    return results, best_model_name

# Usage:
# results, best_model_name = evaluate_models(models, X_test, y_test)

### Feature Importance & SHAP Explainability

In [ ]:
def shap_explainability_analysis(best_model, best_model_name, X_test, X_train):
    """
    Generate SHAP explanations for model interpretability
    """
    
    print("\n" + "="*80)
    print("SHAP EXPLAINABILITY ANALYSIS")
    print("="*80)
    
    try:
        # Create SHAP explainer
        if best_model_name in ['XGBoost', 'LightGBM']:
            explainer = shap.TreeExplainer(best_model)
        else:
            explainer = shap.Explainer(best_model.predict, X_train)
        
        # Calculate SHAP values
        shap_values = explainer.shap_values(X_test)
        
        # For binary classification, take class 1 (failure)
        if isinstance(shap_values, list):
            shap_values = shap_values[1]
        
        print(f"✓ SHAP values calculated for {best_model_name}")
        
        # ========== SUMMARY PLOT ==========
        plt.figure(figsize=(12, 6))
        shap.summary_plot(shap_values, X_test, plot_type="bar", max_display=15, show=False)
        plt.title(f'SHAP Feature Importance - {best_model_name}', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()
        
        # ========== BEESWARM PLOT ==========
        plt.figure(figsize=(12, 8))
        shap.summary_plot(shap_values, X_test, plot_type="beeswarm", max_display=15, show=False)
        plt.title(f'SHAP Value Distribution - {best_model_name}', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()
        
        print("\n✓ SHAP visualizations generated successfully")
        
        return explainer, shap_values
    
    except Exception as e:
        print(f"⚠️  SHAP analysis not available: {str(e)}")
        print("Proceeding with alternative feature importance method...")
        
        # Fallback: Model's built-in feature importance
        if hasattr(best_model, 'feature_importances_'):
            importance = best_model.feature_importances_
            feature_names = X_test.columns
            
            importance_df = pd.DataFrame({
                'feature': feature_names,
                'importance': importance
            }).sort_values('importance', ascending=False).head(15)
            
            plt.figure(figsize=(10, 6))
            plt.barh(importance_df['feature'], importance_df['importance'])
            plt.xlabel('Importance Score')
            plt.title(f'Feature Importance - {best_model_name}', fontweight='bold')
            plt.gca().invert_yaxis()
            plt.tight_layout()
            plt.show()
            
            print("\n📊 Top 15 Important Features:")
            print(importance_df)
        
        return None, None

# Usage:
# explainer, shap_values = shap_explainability_analysis(best_model, best_model_name, X_test, X_train)

---
## Phase 6: Risk Scoring & Business Recommendations
**Week 5-6 - Asset Risk Tiers & Actionable Insights**

In [ ]:
def generate_risk_scores(best_model, df_original, X_processed, scaler):
    """
    Generate asset-level risk scores and classify into tiers:
    - High Risk (Top 20%)
    - Medium Risk (20-50%)
    - Low Risk (Bottom 50%)
    """
    
    print("\n" + "="*80)
    print("ASSET RISK SCORING & TIERING")
    print("="*80)
    
    # Score all assets
    failure_probabilities = best_model.predict_proba(X_processed)[:, 1]
    
    # Assign risk tiers
    p75 = np.percentile(failure_probabilities, 75)
    p50 = np.percentile(failure_probabilities, 50)
    
    risk_tiers = []
    for prob in failure_probabilities:
        if prob >= p75:
            risk_tiers.append('High')
        elif prob >= p50:
            risk_tiers.append('Medium')
        else:
            risk_tiers.append('Low')
    
    # Create risk scoring dataframe
    risk_df = pd.DataFrame({
        'failure_probability': failure_probabilities,
        'risk_tier': risk_tiers
    })
    
    # Add original features for context
    if 'asset_type' in df_original.columns:
        risk_df['asset_type'] = df_original['asset_type'].values
    
    # Sort by failure probability (descending)
    risk_df = risk_df.sort_values('failure_probability', ascending=False).reset_index(drop=True)
    
    # ========== RISK TIER DISTRIBUTION ==========
    print("\n📊 Risk Tier Distribution:")
    tier_counts = pd.Series(risk_tiers).value_counts()
    for tier in ['High', 'Medium', 'Low']:
        count = tier_counts.get(tier, 0)
        pct = (count / len(risk_tiers)) * 100
        print(f"   {tier} Risk: {count:,} assets ({pct:.1f}%)")
    
    # ========== VISUALIZATION ==========
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Risk tier pie chart
    tier_counts.plot(kind='pie', ax=axes[0], autopct='%1.1f%%',
                     colors=['#e74c3c', '#f39c12', '#2ecc71'],
                     labels=['High Risk', 'Medium Risk', 'Low Risk'])
    axes[0].set_title('Asset Distribution by Risk Tier', fontweight='bold')
    axes[0].set_ylabel('')
    
    # Failure probability distribution
    axes[1].hist(failure_probabilities, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
    axes[1].axvline(p75, color='red', linestyle='--', label=f'High Risk Threshold ({p75:.3f})')
    axes[1].axvline(p50, color='orange', linestyle='--', label=f'Medium Risk Threshold ({p50:.3f})')
    axes[1].set_xlabel('Failure Probability')
    axes[1].set_ylabel('Count')
    axes[1].set_title('Distribution of Failure Probabilities', fontweight='bold')
    axes[1].legend()
    
    plt.tight_layout()
    plt.show()
    
    # ========== TOP 10 HIGH-RISK ASSETS ==========
    print("\n🔴 Top 10 High-Risk Assets (Priority Maintenance):")
    print(risk_df.head(10).to_string())
    
    print("\n" + "="*80 + "\n")
    
    return risk_df

# Usage:
# risk_df = generate_risk_scores(best_model, df, X_processed, scaler)

In [ ]:
def business_recommendations(risk_df, results, best_model_name):
    """
    Generate actionable business recommendations framed in financial terms
    """
    
    print("\n" + "="*80)
    print("BUSINESS RECOMMENDATIONS & ROI ANALYSIS")
    print("="*80)
    
    high_risk_count = len(risk_df[risk_df['risk_tier'] == 'High'])
    high_risk_prob = risk_df[risk_df['risk_tier'] == 'High']['failure_probability'].mean()
    
    medium_risk_count = len(risk_df[risk_df['risk_tier'] == 'Medium'])
    medium_risk_prob = risk_df[risk_df['risk_tier'] == 'Medium']['failure_probability'].mean()
    
    print(f"\n💡 KEY FINDINGS:")
    print(f"\n   1. HIGH-RISK ASSETS: {high_risk_count:,} assets")
    print(f"      • Average failure probability: {high_risk_prob*100:.1f}%")
    print(f"      • Estimated immediate failures (if no action): {int(high_risk_count * high_risk_prob)}")
    
    print(f"\n   2. MEDIUM-RISK ASSETS: {medium_risk_count:,} assets")
    print(f"      • Average failure probability: {medium_risk_prob*100:.1f}%")
    print(f"      • Estimated failures (without intervention): {int(medium_risk_count * medium_risk_prob)}")
    
    print(f"\n   3. MODEL PERFORMANCE: {best_model_name}")
    print(f"      • F1-Score: {results[best_model_name]['f1']:.4f}")
    print(f"      • Recall: {results[best_model_name]['recall']:.4f} (minimizes missed failures)")
    print(f"      • AUC-ROC: {results[best_model_name]['auc_roc']:.4f}")
    
    # ========== FINANCIAL IMPACT ESTIMATION ==========
    print(f"\n💰 ESTIMATED FINANCIAL IMPACT:")
    print(f"\n   ASSUMPTION: Average cost per unplanned outage = $50,000 (direct + indirect)")
    
    assumed_cost_per_outage = 50000
    baseline_failures = int(high_risk_count * high_risk_prob) + int(medium_risk_count * medium_risk_prob)
    baseline_cost = baseline_failures * assumed_cost_per_outage
    
    # Assume 15-25% reduction with proactive maintenance (industry benchmark)
    prevented_failures_low = int(baseline_failures * 0.15)
    prevented_failures_mid = int(baseline_failures * 0.20)
    prevented_failures_high = int(baseline_failures * 0.25)
    
    print(f"\n   Baseline (Reactive): {baseline_failures} failures → ${baseline_cost:,} cost")
    print(f"\n   With Proactive Maintenance (This Model):")
    print(f"      • Conservative (15% reduction): {prevented_failures_low} failures prevented")
    print(f"        → Savings: ${prevented_failures_low * assumed_cost_per_outage:,}")
    print(f"\n      • Mid-range (20% reduction): {prevented_failures_mid} failures prevented")
    print(f"        → Savings: ${prevented_failures_mid * assumed_cost_per_outage:,}")
    print(f"\n      • Optimistic (25% reduction): {prevented_failures_high} failures prevented")
    print(f"        → Savings: ${prevented_failures_high * assumed_cost_per_outage:,}")
    
    # ========== ACTION PLAN ==========
    print(f"\n" + "="*50)
    print(f"📋 RECOMMENDED ACTION PLAN:")
    print(f"="*50)
    
    print(f"\n🔴 IMMEDIATE (WEEK 1-2): HIGH-RISK ASSETS")
    print(f"   Target: {high_risk_count:,} assets with failure probability > {risk_df[risk_df['risk_tier']=='High']['failure_probability'].min():.1%}")
    print(f"   Actions:")
    print(f"      • Intensive condition-based inspections")
    print(f"      • Schedule preventive maintenance within 2 weeks")
    print(f"      • Deploy real-time monitoring sensors on critical assets")
    print(f"      • Budget allocation: ~{(high_risk_count * 5000):,} (assuming $5k/asset for intervention)")
    
    print(f"\n🟠 SHORT-TERM (WEEK 3-8): MEDIUM-RISK ASSETS")
    print(f"   Target: {medium_risk_count:,} assets with medium failure probability")
    print(f"   Actions:")
    print(f"      • Standard maintenance scheduling")
    print(f"      • Quarterly condition assessments")
    print(f"      • Budget allocation: ~{(medium_risk_count * 2000):,} (assuming $2k/asset)")
    
    print(f"\n🟢 ONGOING: LOW-RISK ASSETS")
    print(f"   • Standard annual maintenance")
    print(f"   • Routine inspections")
    print(f"   • Quarterly model retraining to catch emerging risks")
    
    print(f"\n" + "="*80 + "\n")

# Usage:
# business_recommendations(risk_df, results, best_model_name)

---
## Main Execution Pipeline
**Run this cell to execute the complete analysis workflow**

In [ ]:
# ====================================================================
# COMPLETE ANALYSIS PIPELINE - RUN THIS CELL FOR FULL WORKFLOW
# ====================================================================

def run_complete_pipeline(csv_file_path):
    """
    Execute the complete PowerGrid analysis pipeline from data loading to recommendations
    """
    
    print("\n" + "#"*80)
    print("# POWERGRID UTILITY INTELLIGENCE - COMPLETE ANALYSIS PIPELINE")
    print("#"*80)
    
    # ========== PHASE 1: LOAD & PROFILE DATA ==========
    print("\n[PHASE 1] Loading and profiling dataset...")
    df = pd.read_csv(csv_file_path)
    data_profiling_report(df)
    
    # ========== PHASE 2: EXPLORATORY DATA ANALYSIS ==========
    print("\n[PHASE 2] Performing exploratory data analysis...")
    exploratory_data_analysis(df)
    
    # ========== PHASE 3: PREPROCESSING & FEATURE ENGINEERING ==========
    print("\n[PHASE 3] Preprocessing and engineering features...")
    X_balanced, y_balanced, scaler = preprocess_and_engineer_features(df)
    
    # ========== PHASE 4: MODEL TRAINING ==========
    print("\n[PHASE 4] Training multiple classifiers...")
    models, X_train, X_test, y_train, y_test = train_models(X_balanced, y_balanced)
    
    # ========== PHASE 5: EVALUATION & EXPLAINABILITY ==========
    print("\n[PHASE 5] Evaluating models and generating explanations...")
    results, best_model_name = evaluate_models(models, X_test, y_test)
    best_model = models[best_model_name]
    
    # ========== PHASE 5B: SHAP ANALYSIS ==========
    explainer, shap_values = shap_explainability_analysis(best_model, best_model_name, X_test, X_train)
    
    # ========== PHASE 6: RISK SCORING ==========
    print("\n[PHASE 6] Generating asset risk scores...")
    risk_df = generate_risk_scores(best_model, df, X_balanced, scaler)
    
    # ========== PHASE 6B: BUSINESS RECOMMENDATIONS ==========
    business_recommendations(risk_df, results, best_model_name)
    
    print("\n" + "#"*80)
    print("# ✅ COMPLETE ANALYSIS FINISHED SUCCESSFULLY")
    print("#"*80)
    
    return {
        'df_original': df,
        'X_balanced': X_balanced,
        'y_balanced': y_balanced,
        'models': models,
        'best_model': best_model,
        'best_model_name': best_model_name,
        'results': results,
        'risk_df': risk_df,
        'scaler': scaler,
        'explainer': explainer,
        'shap_values': shap_values
    }

# ====================================================================
# EXECUTE WITH YOUR DATA FILE
# ====================================================================
# Replace 'your_powergrid_data.csv' with your actual file path
# pipeline_results = run_complete_pipeline('your_powergrid_data.csv')

---
## Utilities & Export Functions

In [ ]:
def export_risk_scores_to_csv(risk_df, output_filename='powergrid_asset_risk_scores.csv'):
    """
    Export risk scores to CSV for operational teams
    """
    risk_df.to_csv(output_filename, index=False)
    print(f"✓ Risk scores exported to: {output_filename}")

def save_model(best_model, model_filename='powergrid_failure_model.pkl'):
    """
    Save trained model for production deployment
    """
    import pickle
    with open(model_filename, 'wb') as f:
        pickle.dump(best_model, f)
    print(f"✓ Model saved to: {model_filename}")

def generate_summary_report(risk_df, results, best_model_name, output_filename='powergrid_analysis_summary.txt'):
    """
    Generate text summary report of the analysis
    """
    with open(output_filename, 'w') as f:
        f.write("="*80 + "\n")
        f.write("POWERGRID UTILITY INTELLIGENCE - ANALYSIS SUMMARY\n")
        f.write("="*80 + "\n\n")
        
        f.write(f"Best Model: {best_model_name}\n")
        f.write(f"F1-Score: {results[best_model_name]['f1']:.4f}\n")
        f.write(f"Recall: {results[best_model_name]['recall']:.4f}\n")
        f.write(f"AUC-ROC: {results[best_model_name]['auc_roc']:.4f}\n\n")
        
        f.write("Risk Tier Distribution:\n")
        tier_counts = risk_df['risk_tier'].value_counts()
        for tier in ['High', 'Medium', 'Low']:
            count = tier_counts.get(tier, 0)
            f.write(f"  {tier}: {count:,}\n")
    
    print(f"✓ Summary report saved to: {output_filename}")

print("✓ Utility functions loaded successfully!")

---
## Next Steps & Instructions

### 🚀 How to Use This Notebook:

1. **Prepare your data file** (PowerGrid dataset with 50,500 records × 32 features)
   - Ensure it's in CSV format
   - Required columns: grid_failure_flag (target), plus the 31 features

2. **Load the data** in the "Data Loading" cell
   ```python
   df = pd.read_csv('path/to/your/powergrid_data.csv')
   ```

3. **Run the complete pipeline** by executing the "Main Execution Pipeline" cell
   ```python
   pipeline_results = run_complete_pipeline('your_powergrid_data.csv')
   ```

4. **Export results** for operational use
   ```python
   export_risk_scores_to_csv(pipeline_results['risk_df'])
   save_model(pipeline_results['best_model'])
   ```

### 📊 Deliverables Generated:
- ✅ Data profiling report
- ✅ EDA visualizations and insights
- ✅ 4 trained classification models
- ✅ Performance metrics and confusion matrices
- ✅ SHAP explainability analysis
- ✅ Asset-level risk scores (High/Medium/Low)
- ✅ Business recommendations & ROI analysis
- ✅ Risk scores CSV export
- ✅ Trained model pickle file

### 🔧 Dependencies:
```bash
pip install pandas numpy scikit-learn xgboost lightgbm shap matplotlib seaborn scipy
```

---

**Questions or Issues?** Refer back to the relevant phase section or the business recommendations for context.
